# Multi-Class Classification on Synthetic Blobs

Train an MLP to separate K isotropic Gaussian clusters using CrossEntropyLoss.

We use 4 clusters in 2D space — the model must learn non-linear decision
boundaries that partition the plane into K regions.

This is the multi-class counterpart to the binary moons demo.
The key difference: **no Sigmoid at the output** — CrossEntropyLoss
expects raw logits and fuses Softmax + NLL internally.

In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parent) if Path.cwd().name == "apps" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.append(project_root)

import matplotlib.pyplot as plt
import numpy as np
from core.autograd import Value
from core.nn import (
    SGD,
    CrossEntropyLoss,
    DataLoader,
    Linear,
    ReLU,
    Sequential,
)

## 1. Generate Blobs Dataset

Create `n_classes = 4` isotropic Gaussian clusters in 2D.
Each cluster has a distinct center and the same covariance (identity).

In [ ]:
np.random.seed(42)

n_classes = 4
n_per_class = 100
n_samples = n_classes * n_per_class

# Place K cluster centers on a circle of radius 2.0
R = 2.0
angles = np.linspace(0, 2 * np.pi, n_classes, endpoint=False)
centers = np.stack([R * np.cos(angles), R * np.sin(angles)], axis=1)

# Generate points for each cluster
X_list, y_list = [], []
for k in range(n_classes):
    cluster_x = centers[k] + 0.4 * np.random.randn(n_per_class, 2)
    X_list.append(cluster_x)
    y_list.append(np.full(n_per_class, k))

X = np.vstack(X_list)
y = np.concatenate(y_list)

# Shuffle and split 80/20
indices = np.random.permutation(n_samples)
split = int(0.8 * n_samples)
x_train, x_val = X[indices[:split]], X[indices[split:]]
y_train, y_val = y[indices[:split]], y[indices[split:]]

print(f"Train: {x_train.shape}, Val: {x_val.shape}")
print(f"Classes: {np.unique(y_train)}")

In [ ]:
# Quick look at the data
colors = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3"]
for k in range(n_classes):
    mask = y_train == k
    plt.scatter(
        x_train[mask, 0],
        x_train[mask, 1],
        s=10,
        alpha=0.7,
        color=colors[k],
        label=f"class {k}",
    )
plt.legend()
plt.title(f"Blobs Dataset ({n_classes} classes)")
plt.gca().set_aspect("equal")

## 2. Model Definition

Architecture: `Linear(2, 32) -> ReLU() -> Linear(32, 32) -> ReLU() -> Linear(32, n_classes)`

**Key difference from binary classification**:
- The output layer has `n_classes` neurons (one per class)
- **No Sigmoid** at the output — CrossEntropyLoss fuses Softmax internally
- These raw outputs are called **logits**

In [ ]:
model = Sequential(
    [Linear(2, 32), ReLU(), Linear(32, 32), ReLU(), Linear(32, n_classes)]
)

print(model)

## 3. Loss & Optimizer

CrossEntropyLoss takes **raw logits** and integer class labels.
It fuses Softmax + NLL into one numerically stable operation.

In [ ]:
loss_fn = CrossEntropyLoss()
optim = SGD(model.parameters(), lr=0.001)

## 4. Training Loop

Same training loop pattern as before — forward, backward, step.
The `target` Value passed to CrossEntropyLoss holds integer class indices, not one-hot.

In [ ]:
train_loader = DataLoader(x_train, y_train.reshape(-1, 1), batch_size=16, shuffle=True)

epochs = 500
train_losses = []
val_losses = []

for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_x, batch_y in train_loader:
        bx, by = Value(batch_x), Value(batch_y)

        logits = model(bx)
        loss = loss_fn(logits, by)

        optim.zero_grad()
        loss.backward()
        optim.step()

        epoch_loss += loss.data

    train_losses.append(epoch_loss / len(train_loader))

    vx = Value(x_val)
    vy = Value(y_val.reshape(-1, 1))
    vlogits = model(vx)
    vloss = loss_fn(vlogits, vy)
    val_losses.append(vloss.data)

    if (epoch + 1) % 100 == 0:
        print(
            f"Epoch {epoch + 1:4d} | train loss: {train_losses[-1]:.4f} | val loss: {val_losses[-1]:.4f}"
        )

print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final val loss:   {val_losses[-1]:.4f}")

## 5. Loss Curves

In [ ]:
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.legend()
plt.title("Training Progress")

## 6. Decision Regions

The multi-class decision boundary partitions the plane into K regions.
A point is assigned to the class with the highest logit (argmax).

In [ ]:
model.eval()

# Create a meshgrid covering the data extent with some margin
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
h = 0.02  # grid step size
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Evaluate model on every grid point
grid_pts = Value(np.c_[xx.ravel(), yy.ravel()])
logits = model(grid_pts).data  # (N_grid, n_classes)
preds = np.argmax(logits, axis=1)  # (N_grid,)  class index
preds = preds.reshape(xx.shape)  # 2D grid shape

# Plot decision regions
colors = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3"]
plt.contourf(xx, yy, preds, levels=[-0.5, 0.5, 1.5, 2.5, 3.5], colors=colors, alpha=0.3)

# Scatter training points on top
for k in range(n_classes):
    mask = y_train == k
    plt.scatter(
        x_train[mask, 0],
        x_train[mask, 1],
        s=10,
        alpha=0.7,
        color=colors[k],
        label=f"class {k}",
    )
plt.legend()
plt.title("Decision Regions")
plt.gca().set_aspect("equal")

## 7. Accuracy

Compute classification accuracy.
`logits.argmax(axis=1)` gives the predicted class index.

In [ ]:
pred = model(vx)
pred_label = pred.data.argmax(axis=1)
accuracy = (pred_label == y_val).mean()
print(f"Validation accuracy: {accuracy * 100:.1f}%")